This notebook attempts to load existing trained siamese backbones, add a new classification head, then attempt to tune for classification.

In [1]:
# Import helper code
import sys
sys.path.insert(1, '../')
import helpers
import torch

In [18]:
import os
dataset_path = "../split_node21_sets/"
process = "arch_seg" #"lung_seg" "crop" 
train_set = "chestxray14"
bsz = 64
resize_dim = 224
# run_index = 0
# best_epoch_to_load = 64
# run_index = 1
# best_epoch_to_load = 72
run_index = 2
best_epoch_to_load = 59
weights = torch.load(f'logs/subsets/chestxray14/rad_unfrz_cosine_{process}_{run_index}/checkpoints/checkpoint_epoch_{best_epoch_to_load}.pth')
# weights = torch.load('logs/subsets/chestxray14/rad_unfrz_cosine_crop_0/checkpoints/best_model.pth',map_location=device)

from torchvision import transforms
import torch
base_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.ToTensor(),
])

augment_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])
# Load with default settings
train_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "train"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=augment_transform,
    cache_in_ram=True

)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  4.34it/s]


Caching base images into RAM...


Caching: 100%|██████████| 2478/2478 [00:14<00:00, 165.98it/s]

Cached 2478 images
Total pairs: 1239
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 420
  normal (idx=0): 819


In [3]:
test_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  8.58it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1062/1062 [00:06<00:00, 163.18it/s]

Cached 1062 images
Total pairs: 531
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 180
  normal (idx=0): 351


In [4]:
test2_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,"padchest", "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 15.48it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1008/1008 [00:06<00:00, 163.83it/s]

Cached 1008 images
Total pairs: 504
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 94
  normal (idx=0): 410


In [5]:
test3_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,"jsrt", "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 113.50it/s]


Caching base images into RAM...


Caching: 100%|██████████| 144/144 [00:00<00:00, 161.89it/s]

Cached 144 images
Total pairs: 72
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 44
  normal (idx=0): 28


## Create Classifier Head

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiameseClassifier(nn.Module):
    """
    Wraps trained SiameseNetwork with a classification head
    Default freeze the Siamese backbone for fast fine-tuning
    """
    def __init__(self, siamese_model, embedding_dim=128, freeze_siamese=True):
        super(SiameseClassifier, self).__init__()
        
        self.siamese = siamese_model
        
        # Freeze the siamese network if desired
        if freeze_siamese:
            for param in self.siamese.parameters():
                param.requires_grad = False
        
        # Classification head operates on distance/concatenated embeddings
        # Use both distance + embeddings
        
        input_dim = 2 * embedding_dim + 1  # concatenated embeddings + distance
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)  # Binary classification (sigmoid applied later)
        )
        
    def forward(self, x1, x2, return_embeddings=False, distance_metric='cosine'):
        """
        Args:
            x1, x2: Input image pairs
            return_embeddings: If True, also return embeddings for analysis
            distance_metric: 'euclidean' or 'cosine'
        Returns:
            logits: Classification logits (before sigmoid)
            (optional) emb1, emb2, distance
        """
        # Get embeddings from siamese network
        emb1, emb2 = self.siamese(x1, x2)
        
        # Calculate distance
        if distance_metric == 'euclidean':
            distance = F.pairwise_distance(emb1, emb2, p=2)
        elif distance_metric == 'cosine':
            cosine_sim = torch.sum(emb1 * emb2, dim=1)
            distance = 1 - cosine_sim
        else:
            raise ValueError(f"Unknown distance metric: {distance_metric}")
        
        # Concatenate embeddings and distance
        # Shape: (batch, 2*embedding_dim + 1)
        combined = torch.cat([emb1, emb2, distance.unsqueeze(1)], dim=1)
        
        # Get classification logits
        logits = self.classifier(combined).squeeze()
        
        if return_embeddings:
            return logits, emb1, emb2, distance
        return logits

In [20]:
# import training metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

In [21]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [22]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'rad_cxr14_cosine_{process}_{run_index}_best_classifier_f1_1.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5743 | Acc: 0.8491 | Prec: 0.7325 | Rec: 0.8738 | F1: 0.7970 | AUC: 0.9204
  Test   Loss: 0.4745 | Acc: 0.8324 | Prec: 0.9333 | Rec: 0.5444 | F1: 0.6877 | AUC: 0.8635
  Padchest   Loss: 0.4123 | Acc: 0.8631 | Prec: 0.7907 | Rec: 0.3617 | F1: 0.4964 | AUC: 0.7953
  JSRT   Loss: 0.6325 | Acc: 0.5556 | Prec: 1.0000 | Rec: 0.2727 | F1: 0.4286 | AUC: 0.7524
  ✓ Saved new best model! (F1: 0.6877)
Epoch 2/20
  Train Loss: 0.2385 | Acc: 0.9500 | Prec: 0.9891 | Rec: 0.8619 | F1: 0.9211 | AUC: 0.9839
  Test   Loss: 0.4747 | Acc: 0.8399 | Prec: 0.8519 | Rec: 0.6389 | F1: 0.7302 | AUC: 0.8550
  Padchest   Loss: 0.3914 | Acc: 0.8611 | Prec: 0.6765 | Rec: 0.4894 | F1: 0.5679 | AUC: 0.7863
  JSRT   Loss: 1.2816 | Acc: 0.6250 | Prec: 1.0000 | Rec: 0.3864 | F1: 0.5574 | AUC: 0.7476
  ✓ Saved new best model! (F1: 0.7302)
Epoch 3/20
  Train Loss: 0.0839 | Acc: 0.9790 | Prec: 0.9782 | Rec: 0.9595 | F1: 0.9688 | AUC: 0.9908
  Test   Loss: 0.5374 | Acc: 0.8380 | Prec: 0.8176 | Rec

In [23]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [24]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'rad_cxr14_cosine_{process}_{run_index}_best_classifier_f1_2.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5842 | Acc: 0.7554 | Prec: 1.0000 | Rec: 0.2786 | F1: 0.4358 | AUC: 0.9383
  Test   Loss: 0.4898 | Acc: 0.8098 | Prec: 0.8835 | Rec: 0.5056 | F1: 0.6431 | AUC: 0.8610
  Padchest   Loss: 0.4306 | Acc: 0.8631 | Prec: 0.8049 | Rec: 0.3511 | F1: 0.4889 | AUC: 0.7941
  JSRT   Loss: 0.8830 | Acc: 0.4722 | Prec: 1.0000 | Rec: 0.1364 | F1: 0.2400 | AUC: 0.7492
  ✓ Saved new best model! (F1: 0.6431)
Epoch 2/20
  Train Loss: 0.2391 | Acc: 0.9532 | Prec: 0.9866 | Rec: 0.8738 | F1: 0.9268 | AUC: 0.9877
  Test   Loss: 0.4668 | Acc: 0.8437 | Prec: 0.8540 | Rec: 0.6500 | F1: 0.7382 | AUC: 0.8543
  Padchest   Loss: 0.3968 | Acc: 0.8651 | Prec: 0.6912 | Rec: 0.5000 | F1: 0.5802 | AUC: 0.7835
  JSRT   Loss: 0.9161 | Acc: 0.6250 | Prec: 1.0000 | Rec: 0.3864 | F1: 0.5574 | AUC: 0.7443
  ✓ Saved new best model! (F1: 0.7382)
Epoch 3/20
  Train Loss: 0.0710 | Acc: 0.9790 | Prec: 0.9782 | Rec: 0.9595 | F1: 0.9688 | AUC: 0.9974
  Test   Loss: 0.6453 | Acc: 0.8437 | Prec: 0.8170 | Rec

In [25]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [26]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'rad_cxr14_cosine_{process}_{run_index}_best_classifier_f1_3.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5958 | Acc: 0.7441 | Prec: 0.9905 | Rec: 0.2476 | F1: 0.3962 | AUC: 0.9114
  Test   Loss: 0.5279 | Acc: 0.8079 | Prec: 0.9239 | Rec: 0.4722 | F1: 0.6250 | AUC: 0.8614
  Padchest   Loss: 0.4496 | Acc: 0.8591 | Prec: 0.7805 | Rec: 0.3404 | F1: 0.4741 | AUC: 0.7897
  JSRT   Loss: 0.7516 | Acc: 0.5139 | Prec: 1.0000 | Rec: 0.2045 | F1: 0.3396 | AUC: 0.7638
  ✓ Saved new best model! (F1: 0.6250)
Epoch 2/20
  Train Loss: 0.2657 | Acc: 0.9483 | Prec: 0.9811 | Rec: 0.8643 | F1: 0.9190 | AUC: 0.9832
  Test   Loss: 0.4567 | Acc: 0.8399 | Prec: 0.8571 | Rec: 0.6333 | F1: 0.7284 | AUC: 0.8532
  Padchest   Loss: 0.3851 | Acc: 0.8651 | Prec: 0.6970 | Rec: 0.4894 | F1: 0.5750 | AUC: 0.7852
  JSRT   Loss: 1.3630 | Acc: 0.6111 | Prec: 1.0000 | Rec: 0.3636 | F1: 0.5333 | AUC: 0.7589
  ✓ Saved new best model! (F1: 0.7284)
Epoch 3/20
  Train Loss: 0.0894 | Acc: 0.9750 | Prec: 0.9756 | Rec: 0.9500 | F1: 0.9626 | AUC: 0.9921
  Test   Loss: 0.5823 | Acc: 0.8418 | Prec: 0.8380 | Rec